# Simple example notebook

Simple standlone running script for fitting a CLDS with $u_t = t$, from the README

In [2]:
import os; os.environ["JAX_PLATFORMS"] = "cpu"
import jax.numpy as jnp
from jax import random
from clds import utils
from clds.models import CLDS

In [3]:
N = 96      # Emissions dim
M = 1       # Conditions dim
D = 2       # Latent state dim

T = 100     # N timesteps
B = 8       # N trials

# Data: emissions (B, T, N), conditions (B, T, M)
emissions  = random.normal(random.key(0), shape=(B, T, N))
time = jnp.linspace(0.0, 1.0, T)
conditions = jnp.broadcast_to(time[None, :, None], (B, T, 1))

In [5]:
# Initialize CLDS 
basis = utils.initialize_basis(conditions, lengthscale=0.1)
model = CLDS(
    state_dim=D, 
    emission_dim=N,
    basis=basis,
    gp_priors=["A", "b"],       # A(u), b(u)
    use_dynamics_bias=True,     # Include b
    use_emissions_bias=False,   # Exclude d
    )

In [6]:
# Fit (optional: mask of shape (B, T,))
params, (log_liks, _) = model.fit(
    emissions, conditions, 
    num_iters=100, 
    data_driven_init=False, # data is fake, no need to try PPCA/subspace-ID.
    )

  0%|          | 0/100 [00:00<?, ?it/s]